## 1. Load configuration

Paths and settings are stored in `config/dev.yaml` and `config/prod.yaml`, not in the code. Change `ENV` to `"prod"` to run on the full dataset.

In [1]:
import os, sys
sys.path.append("../src")          # lets the notebook import files from src/

from config_loader import load_config

ENV = os.environ.get("PIPELINE_ENV", "prod")   # "dev" = 2,000-row sample, "prod" = full data
config = load_config(f"../config/{ENV}.yaml")

print("Environment:", ENV)
print(config)

Environment: prod
{'environment': 'prod', 'spark': {'driver_memory': '4g', 'cores': 2, 'shuffle_partitions': 400}, 'source': {'dataset_name': 'ems_incident_dispatch', 'dataset_id': '76xm-jjuj', 'input_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/EMS_Incident_Dispatch_Data.csv'}, 'bronze': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/bronze'}, 'silver': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/silver'}, 'gold': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/gold'}, 'collisions': {'dataset_name': 'motor_vehicle_collisions', 'dataset_id': 'h9gi-nx95', 'input_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/collisions_data.csv'}}


## 2. Start Spark

Create the Spark session, which is the entry point for all Spark operations. Memory, cores and shuffle partitions come from the config. `getOrCreate()` reuses a running session, so **restart the kernel** after changing these settings.

In [2]:
from pyspark.sql import SparkSession

spark_cfg = config["spark"]

spark = (
    SparkSession.builder
    .appName("ems_pipeline")
    .master(f"local[{spark_cfg['cores']}]")
    .config("spark.driver.memory", spark_cfg["driver_memory"])
    .config("spark.sql.shuffle.partitions", spark_cfg["shuffle_partitions"])
    .getOrCreate()
)

print(spark.version)
print("Master:", spark.sparkContext.master)
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 12:00:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/14 12:00:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


3.5.0
Master: local[2]
Driver memory: 4g


## Performance measurement helper

Spark is lazy: transformations like `withColumn` take no time, because the work only happens at an **action** (`count`, `collect`, `write`). We wrap the actions in `timed(...)`, and each printed time covers the whole chain of work behind that action.

In [3]:
import time
from contextlib import contextmanager

@contextmanager
def timed(label):
    start = time.time()
    yield
    print(f"[{label}] {time.time() - start:.1f} s")

## 3. Read the latest collisions Bronze snapshot

In [4]:
import os, glob

bronze_root = os.path.abspath(os.path.join(
    "..", config["bronze"]["root_path"], config["collisions"]["dataset_name"]
))
snapshots = sorted(glob.glob(os.path.join(bronze_root, "snapshot_date=*")))
print("Snapshots found:", [os.path.basename(s) for s in snapshots])

latest_snapshot = snapshots[-1]
bronze_df = spark.read.parquet(latest_snapshot)
total_rows = bronze_df.count()
print("Using:", latest_snapshot)
print("Rows:", total_rows)

Snapshots found: ['snapshot_date=2026-11-14']


Using: /home/ubuntu/spark-notebooks/ems-pipeline/data/bronze/motor_vehicle_collisions/snapshot_date=2026-11-14
Rows: 2301383


## 4. Profile the raw collision data

In [5]:
import pandas as pd
from pyspark.sql import functions as F

cols = [c for c in bronze_df.columns if not c.startswith("_")]
d = F.to_date(F.col("CRASH_DATE"), "MM/dd/yyyy")

with timed("Collision profile"):
    nulls = bronze_df.agg(*[
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c) for c in cols
    ]).collect()[0].asDict()
    prof = bronze_df.agg(
        F.min(d).alias("first_date"), F.max(d).alias("last_date"),
        F.sum(F.when(d.isNull(), 1).otherwise(0)).alias("unparseable_dates"),
        F.countDistinct("COLLISION_ID").alias("distinct_ids"), F.count("*").alias("rows"),
    ).collect()[0].asDict()

print(prof)
profile = pd.DataFrame({"column": cols, "missing": [nulls[c] for c in cols]})
profile["missing_pct"] = (100 * profile["missing"] / total_rows).round(1)
display(profile.sort_values("missing", ascending=False).head(12))

bronze_df.groupBy("BOROUGH").count().orderBy(F.desc("count")).show()
bronze_df.select("CRASH_DATE", "CRASH_TIME", "ZIP_CODE", "LATITUDE").show(5)

26/11/14 12:01:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[Collision profile] 9.0 s
{'first_date': datetime.date(2012, 7, 1), 'last_date': datetime.date(2026, 10, 26), 'unparseable_dates': 0, 'distinct_ids': 2301383, 'rows': 2301383}


,column,missing,missing_pct
28,VEHICLE_TYPE_CODE_5,2291215,99.6
22,CONTRIBUTING_FACTOR_VEHICLE_5,2290876,99.5
27,VEHICLE_TYPE_CODE_4,2264502,98.4
21,CONTRIBUTING_FACTOR_VEHICLE_4,2263059,98.3
26,VEHICLE_TYPE_CODE_3,2141174,93.0
20,CONTRIBUTING_FACTOR_VEHICLE_3,2134293,92.7
8,CROSS_STREET_NAME,1884202,81.9
9,OFF_STREET_NAME,885213,38.5
3,ZIP_CODE,699161,30.4
2,BOROUGH,674806,29.3


+-------------+------+
|      BOROUGH| count|
+-------------+------+
|         NULL|674806|
|     BROOKLYN|521443|
|       QUEENS|437651|
|    MANHATTAN|356811|
|        BRONX|242873|
|STATEN ISLAND| 67799|
+-------------+------+

+----------+----------+--------+---------+
|CRASH_DATE|CRASH_TIME|ZIP_CODE| LATITUDE|
+----------+----------+--------+---------+
|10/26/2026|     15:20|   11372|40.749653|
|10/26/2026|     16:51|   11212|40.661663|
|10/26/2026|     19:30|   11432|40.706635|
|10/26/2026|     05:45|    NULL| 40.77529|
|10/26/2026|     10:00|   11215| 40.67216|
+----------+----------+--------+---------+
only showing top 5 rows



## 5. Silver transformations (rules in src/collisions_silver_rules.py; cheap one-column duplicate check first)

In [6]:
from collisions_silver_rules import (
    convert_types, split_rejected, dedupe_by_collision_id, flag_issues,
)

typed_df = convert_types(bronze_df)
accepted_df, rejected_df = split_rejected(typed_df)

with timed("Duplicate check (COLLISION_ID only)"):
    rows_accepted = accepted_df.count()
    distinct_ids = accepted_df.select("COLLISION_ID").distinct().count()
dup_count = rows_accepted - distinct_ids
print("Accepted rows:", rows_accepted, "| Duplicate collision IDs:", dup_count)

deduped_df = dedupe_by_collision_id(accepted_df) if dup_count > 0 else accepted_df
clean_df = flag_issues(deduped_df)
rows_in = total_rows

[Duplicate check (COLLISION_ID only)] 4.4 s
Accepted rows: 2301383 | Duplicate collision IDs: 0


## 6. Write the Silver outputs (Parquet partitioned by collision_year, one file per year; rejected records saved separately)

In [7]:
silver_root = os.path.abspath(os.path.join(
    "..", config["silver"]["root_path"], config["collisions"]["dataset_name"]
))
silver_data_path = os.path.join(silver_root, "data")
rejected_path = os.path.join(silver_root, "rejected")
bronze_snapshot_date = os.path.basename(latest_snapshot).split("=")[1]

silver_out = (
    clean_df
    .withColumn("_bronze_snapshot_date", F.lit(bronze_snapshot_date))
    .withColumn("_silver_processed_at", F.current_timestamp())
)
rejected_out = rejected_df.withColumn("_reject_reason", F.lit("missing collision id or unparseable date"))

with timed("Silver write"):
    (silver_out
        .repartition("collision_year")
        .write.mode("overwrite")
        .partitionBy("collision_year")
        .parquet(silver_data_path))

rejected_out.write.mode("overwrite").parquet(rejected_path)
print("Silver table written to:", silver_data_path)

[Silver write] 19.6 s


Silver table written to: /home/ubuntu/spark-notebooks/ems-pipeline/data/silver/motor_vehicle_collisions/data


## 7. Quality summary (computed from the written Silver data)

In [8]:
silver_check = spark.read.parquet(silver_data_path)

with timed("Quality summary"):
    rows_rejected = rows_in - rows_accepted
    rows_out = silver_check.count()
    flags = silver_check.agg(
        F.sum(F.when(~F.col("has_valid_location"), 1).otherwise(0)).alias("no_valid_location"),
        F.sum(F.when(F.col("has_data_issue"), 1).otherwise(0)).alias("data_issue"),
        F.sum(F.when(F.col("borough_std") == "UNKNOWN", 1).otherwise(0)).alias("unknown_borough"),
    ).collect()[0].asDict()

with timed("Grain check on written Silver"):
    distinct_out = silver_check.select("COLLISION_ID").distinct().count()

assert rows_out == rows_in - rows_rejected - dup_count, "Row counts do not add up!"
assert distinct_out == rows_out, "Silver has duplicate COLLISION_IDs!"

quality_summary = pd.DataFrame([
    ("Rows in (Bronze)", rows_in),
    ("Rejected (no ID or unparseable date)", rows_rejected),
    ("Duplicate collision IDs removed", dup_count),
    ("Rows out (Silver)", rows_out),
    ("Flagged: no valid location", flags["no_valid_location"]),
    ("Flagged: data issue", flags["data_issue"]),
    ("Borough UNKNOWN (kept)", flags["unknown_borough"]),
], columns=["metric", "count"])
quality_summary

[Quality summary] 0.4 s


[Grain check on written Silver] 2.4 s


,metric,count
0,Rows in (Bronze),2301383
1,Rejected (no ID or unparseable date),0
2,Duplicate collision IDs removed,0
3,Rows out (Silver),2301383
4,Flagged: no valid location,249289
5,Flagged: data issue,0
6,Borough UNKNOWN (kept),674806


## 8. Save the quality summary

In [9]:
import json
from datetime import datetime, timezone

quality_dir = os.path.join(silver_root, "_quality_log")
os.makedirs(quality_dir, exist_ok=True)
quality_record = {
    "environment": config["environment"],
    "bronze_snapshot_date": bronze_snapshot_date,
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "metrics": {row["metric"]: int(row["count"]) for _, row in quality_summary.iterrows()},
}
with open(os.path.join(quality_dir, f"silver_quality_{bronze_snapshot_date}.json"), "w") as f:
    json.dump(quality_record, f, indent=2)
print(json.dumps(quality_record, indent=2))

{
  "environment": "prod",
  "bronze_snapshot_date": "2026-11-14",
  "processed_at_utc": "2026-11-14T12:15:48.679337+00:00",
  "metrics": {
    "Rows in (Bronze)": 2301383,
    "Rejected (no ID or unparseable date)": 0,
    "Duplicate collision IDs removed": 0,
    "Rows out (Silver)": 2301383,
    "Flagged: no valid location": 249289,
    "Flagged: data issue": 0,
    "Borough UNKNOWN (kept)": 674806
  }
}
